# Phase 4.4 — Baseline XGBoost Shot Make Prediction

**Goal:** Train a baseline XGBoost classifier that predicts `P(make)` — the probability that a shot will be made — from shot context features.

**Dataset:** `data/processed/cleaned_shot_logs.csv` — 128,069 NBA shot attempts.

**Target:** `shot_made` (1 = made, 0 = missed).

**Features:** Shot distance, shot angle, defender distance, shot value, shot zone (one-hot), and pressure level (one-hot) — the same feature set used by `backend/app/ml/feature_builder.py`.

This is a **first baseline** model. No aggressive hyperparameter tuning. The goal is to establish a working prediction pipeline and understand what features the model finds important.

## 1. Imports

In [ ]:
import sys
import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    roc_curve,
    ConfusionMatrixDisplay,
)
import xgboost as xgb

# Make backend importable from notebooks/
sys.path.insert(0, os.path.join('..', 'backend'))
from app.ml.feature_builder import build_features_from_dataframe, MODEL_FEATURES

print(f"XGBoost version : {xgb.__version__}")
print(f"Model features  : {MODEL_FEATURES}")

## 2. Load Cleaned Data

We use the processed dataset that was cleaned and validated in the previous phase. The raw dataset contains 15 columns; we only need the feature columns plus the target `shot_made`.

In [ ]:
DATA_PATH = os.path.join('..', 'data', 'processed', 'cleaned_shot_logs.csv')

df = pd.read_csv(DATA_PATH)

print(f"Dataset shape  : {df.shape}")
print(f"Columns        : {df.columns.tolist()}")
print(f"\nTarget distribution (shot_made):")
print(df['shot_made'].value_counts())
print(f"\nMake rate: {df['shot_made'].mean():.3f}")

df.head()

## 3. Feature Engineering

We reuse `build_features_from_dataframe()` from `backend/app/ml/feature_builder.py`. This function:

- Passes through numeric features (`shot_distance`, `shot_angle`, `defender_distance`, `shot_value`).
- One-hot encodes `shot_zone` → 4 binary columns (`paint`, `mid_range`, `three_point`, `corner_three`).
- One-hot encodes `pressure_level` → 4 binary columns (`very_tight`, `tight`, `open`, `very_open`).

Using the same function here ensures that the model training and the production API use identical feature representations — no train/serve skew.

In [ ]:
# shot_angle is not in the raw dataset; feature_builder uses df.get() with default 0
# This is intentional for the baseline — angle data can be added later.
X = build_features_from_dataframe(df)
y = df['shot_made'].astype(int)

print(f"Feature matrix shape : {X.shape}")
print(f"Target shape         : {y.shape}")
print(f"\nFeature columns:")
print(X.columns.tolist())
print(f"\nMissing values in features:")
print(X.isnull().sum())

X.head()

## 4. Train / Test Split

80 % training, 20 % test. `stratify=y` preserves the class ratio in both splits so the test set is representative of overall make rate.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print(f"Training samples : {len(X_train):,}")
print(f"Test samples     : {len(X_test):,}")
print(f"Train make rate  : {y_train.mean():.3f}")
print(f"Test  make rate  : {y_test.mean():.3f}")

## 5. Train Baseline XGBoost Model

We use conservative defaults for the first baseline:
- `n_estimators=200` — enough trees to learn patterns without overfitting.
- `max_depth=4` — shallow trees reduce overfitting on tabular data.
- `learning_rate=0.1` — standard starting point.
- `eval_metric='logloss'` — optimises predicted probabilities, not just class labels.
- `use_label_encoder=False` — avoids deprecation warnings in XGBoost 1.6+.

No grid search or cross-validation yet — that is Phase 4.5.

In [ ]:
model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1,
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False,
)

print("Model training complete.")
print(f"Estimators trained: {model.n_estimators}")

## 6. Predict Probabilities

`predict_proba()` returns `[P(miss), P(make)]` for each shot. We take the second column — `P(make)` — as our primary output. This is what the production API will serve to the frontend.

Hard class predictions (`predict()`) are derived by thresholding at 0.5 and are used only for classification metrics.

In [ ]:
y_prob = model.predict_proba(X_test)[:, 1]   # P(make)
y_pred = model.predict(X_test)               # hard class labels (threshold = 0.5)

print(f"Predicted probability range : [{y_prob.min():.3f}, {y_prob.max():.3f}]")
print(f"Mean predicted P(make)      : {y_prob.mean():.3f}")
print(f"Actual make rate (test)     : {y_test.mean():.3f}")

# Sample of predictions vs actuals
sample = pd.DataFrame({
    'actual': y_test.values[:10],
    'P(make)': y_prob[:10].round(3),
    'predicted': y_pred[:10],
})
print("\nSample predictions:")
print(sample.to_string(index=False))

## 7. Evaluation Metrics

We report five metrics:

| Metric | What it measures |
|--------|------------------|
| **Accuracy** | Overall correct predictions |
| **Precision** | Of shots predicted as made, how many actually were |
| **Recall** | Of shots that were actually made, how many did we catch |
| **F1** | Harmonic mean of precision and recall |
| **ROC-AUC** | How well the model separates makes from misses across all thresholds |

For shot prediction, **ROC-AUC** is the most meaningful metric — we care about the model's ranking (higher probability → more likely to go in), not just the hard 0.5 threshold.

In [ ]:
accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall    = recall_score(y_test, y_pred)
f1        = f1_score(y_test, y_pred)
roc_auc   = roc_auc_score(y_test, y_prob)

print("=" * 40)
print("BASELINE XGBOOST — TEST SET METRICS")
print("=" * 40)
print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-score  : {f1:.4f}")
print(f"ROC-AUC   : {roc_auc:.4f}")
print("=" * 40)

print("\nFull classification report:")
print(classification_report(y_test, y_pred, target_names=['Miss', 'Make']))

### 7a. Confusion Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=['Miss', 'Make'],
    cmap='Blues',
    ax=ax,
)
ax.set_title('Confusion Matrix — Baseline XGBoost')
plt.tight_layout()
plt.show()

### 7b. ROC Curve

The ROC curve shows the tradeoff between true positive rate (sensitivity) and false positive rate across all decision thresholds. The area under this curve (AUC) is our primary model quality indicator.

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_prob)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, color='steelblue', lw=2, label=f'XGBoost (AUC = {roc_auc:.3f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random baseline (AUC = 0.500)')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — Baseline XGBoost')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 7c. Probability Distribution

A well-calibrated model should produce different probability distributions for makes and misses — makes should skew toward higher P(make) values and misses toward lower ones.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

ax.hist(y_prob[y_test == 0], bins=40, alpha=0.6, color='tomato',    label='Miss (actual)')
ax.hist(y_prob[y_test == 1], bins=40, alpha=0.6, color='steelblue', label='Make (actual)')

ax.set_xlabel('Predicted P(make)')
ax.set_ylabel('Count')
ax.set_title('Predicted Probability Distribution by Outcome')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Feature Importance

XGBoost provides feature importance scores that tell us how much each feature contributed to the model's decisions. We use `weight` (how often a feature is used in splits) by default, but also show `gain` (average improvement in accuracy per split) which is more informative.

**What we expect the model to learn:**
- `shot_distance` should be the strongest predictor — shots closer to the basket are made more often.
- `defender_distance` should matter — tighter defense reduces make probability.
- `shot_zone_*` features encode discrete court regions and should correlate with historical make rates.
- `pressure_*` features capture the same information as `defender_distance` but in categorical form.

In [ ]:
importance_df = pd.DataFrame({
    'feature': MODEL_FEATURES,
    'importance_weight': model.feature_importances_,
}).sort_values('importance_weight', ascending=False)

print("Feature importances (by split frequency):")
print(importance_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['steelblue' if imp > importance_df['importance_weight'].median() else 'lightsteelblue'
          for imp in importance_df['importance_weight']]
ax.barh(importance_df['feature'], importance_df['importance_weight'], color=colors)
ax.set_xlabel('Feature Importance (weight)')
ax.set_title('XGBoost Feature Importance — Baseline Model')
ax.invert_yaxis()
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Save Model

Save the trained model to `backend/app/ml/models/` so the FastAPI server can load it. We save in XGBoost's native `.ubj` format (JSON-based binary) which is version-stable.

In [ ]:
MODEL_DIR = os.path.join('..', 'backend', 'app', 'ml', 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

MODEL_PATH = os.path.join(MODEL_DIR, 'shot_make_xgb_baseline.ubj')
model.save_model(MODEL_PATH)

print(f"Model saved to: {MODEL_PATH}")

# Quick smoke-test: reload and verify probabilities match
loaded = xgb.XGBClassifier()
loaded.load_model(MODEL_PATH)
assert np.allclose(loaded.predict_proba(X_test[:5])[:, 1], y_prob[:5]), "Reload mismatch!"
print("Reload verification passed.")

## 10. Baseline Summary

| Metric | Value |
|--------|-------|
| Training samples | 102,455 |
| Test samples | 25,614 |
| Features | 12 |
| Trees | 200 |
| Max depth | 4 |

**Interpretation:**

- The model predicts shot make probability from static shot context (distance, zone, defender pressure).
- ROC-AUC > 0.60 indicates the model has learned a genuine signal — it is not just guessing.
- The dominant features are expected to be `shot_distance` and `defender_distance`.
- Shot zone one-hot features partly overlap with `shot_distance` (both encode court position); a future version could drop one set.
- `shot_angle` defaults to 0 for all rows (not present in this dataset) and contributes nothing — it should be added or removed in Phase 4.5.

**Next steps (Phase 4.5):**
- Cross-validate with `StratifiedKFold`.
- Tune `max_depth`, `learning_rate`, `n_estimators` with `Optuna` or `GridSearchCV`.
- Add shot angle from raw coordinate data.
- Evaluate player-level calibration (does Curry get a higher baseline P(make) than an average player?).